In [1]:
# Mount Drive and load pretrained ST-GCN architecture
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import torch
import torch.nn as nn
import math
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

STGCN_DIR = '/content/st-gcn'

# Clone ST-GCN if not present
if not os.path.exists(STGCN_DIR):
    os.system(f'git clone https://github.com/yysijie/st-gcn.git {STGCN_DIR}')
    print("ST-GCN cloned.")
else:
    print("ST-GCN already present.")

# Patch torch.load
stgcn_io = Path(f'{STGCN_DIR}/torchlight/torchlight/io.py')
text = stgcn_io.read_text()
if 'weights_only=False' not in text:
    text = text.replace(
        'torch.load(weights_path)',
        'torch.load(weights_path, weights_only=False, map_location="cpu")'
    )
    stgcn_io.write_text(text)
    print("ST-GCN torch.load patch applied.")

os.system(f'pip install -e {STGCN_DIR}/torchlight -q')

# Download pretrained NTU weights if not present
pretrained_path = '/content/st-gcn/models/st_gcn.ntu-xsub.pt'
if not os.path.exists(pretrained_path):
    print("Downloading pretrained ST-GCN NTU weights...")
    os.system('pip install gdown -q')
    os.system(f'gdown 18pcNj4Bu4Ub7S3YJSsRNJ45XxA4GyaYG -O {pretrained_path}')

sys.path.insert(0, STGCN_DIR)
from net.st_gcn import Model as STGCN

print("Setup complete. Ready for data loading.")

Mounted at /content/drive
Using device: cuda
ST-GCN cloned.
ST-GCN torch.load patch applied.
Setup complete. Ready for data loading.


In [2]:
# Load 70/10/20 data and define JointDropoutDataset
import numpy as np
import pickle
import torch
from torch.utils.data import Dataset, DataLoader

# UPDATED PATHS TO 70/10/20
stgcn_dir = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/HRI30_70_10_20/stgcn_format'
X_train = np.load(os.path.join(stgcn_dir, 'train_data.npy')) # (2058, 3, 150, 25, 1)
X_test = np.load(os.path.join(stgcn_dir, 'test_data.npy'))   # (588, 3, 150, 25, 1)

with open(os.path.join(stgcn_dir, 'train_label.pkl'), 'rb') as f:
    _, y_train = pickle.load(f)
with open(os.path.join(stgcn_dir, 'test_label.pkl'), 'rb') as f:
    _, y_test = pickle.load(f)

y_train = np.array(y_train)
y_test = np.array(y_test)

class JointDropoutDataset(Dataset):
    """ST-GCN training dataset with random joint dropout augmentation."""
    def __init__(self, data, labels, drop_rate=0.20, augment=True):
        self.data = data
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.drop_rate = drop_rate
        self.augment = augment

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        x = self.data[idx].copy() # numpy (3, 150, 25, 1) — .copy() not .clone()
        if self.augment:
            V = x.shape[2] # 25 joints
            n_drop = max(1, int(V * self.drop_rate))
            drop_idx = np.random.choice(V, n_drop, replace=False)
            x[:, :, drop_idx, :] = 0.0
        x_tensor = torch.tensor(x, dtype=torch.float32)
        return x_tensor, self.labels[idx]

# Test set has NO augmentation (augment=False)
train_dataset = JointDropoutDataset(X_train, y_train, drop_rate=0.20, augment=True)
test_dataset = JointDropoutDataset(X_test, y_test, drop_rate=0.00, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

batch_x, batch_y = next(iter(train_loader))
print(f"Batch shape: {batch_x.shape} Labels: {batch_y.shape}")

Batch shape: torch.Size([32, 3, 150, 25, 1]) Labels: torch.Size([32])


In [3]:
# Train ST-GCN with Random Joint Dropout
import torch.optim as optim

# Initialize model with 60 classes, then replace final layer with 30 classes
model = STGCN(in_channels=3, num_class=60, dropout=0.5,
              edge_importance_weighting=True,
              graph_args={'layout': 'ntu-rgb+d', 'strategy': 'spatial'})

# Load pretrained NTU weights
ckpt = torch.load('/content/st-gcn/models/st_gcn.ntu-xsub.pt',
                  map_location='cpu', weights_only=False)
model.load_state_dict(ckpt)

# Replace final layer for 30 HRI30 classes
model.fcn = nn.Conv2d(256, 30, kernel_size=1)
nn.init.normal_(model.fcn.weight, 0, math.sqrt(2./30))
model = model.to(device)

# Split learning rates to protect backbone
backbone_params = [p for n, p in model.named_parameters() if 'fcn' not in n]
head_params = [p for n, p in model.named_parameters() if 'fcn' in n]
optimizer = optim.SGD([{'params': backbone_params, 'lr': 1e-3},
                      {'params': head_params, 'lr': 1e-2}],
                     momentum=0.9, nesterov=True, weight_decay=1e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 40], gamma=0.1)
criterion = nn.CrossEntropyLoss()

# Save to new folder to protect old weights
CKPT_DIR = '/content/drive/MyDrive/HRC_Research/checkpoints/stgcn_hri30_rjd_70_10_20'
os.makedirs(CKPT_DIR, exist_ok=True)
best_acc, best_epoch = 0.0, 0

print("Starting ST-GCN-RJD training (50 epochs)...")
for epoch in range(1, 51):
    model.train()
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx = bx.to(device)
            preds = model(bx).argmax(dim=1).cpu()
            correct += (preds == by).sum().item()
            total += by.size(0)
    acc = 100.0 * correct / total
    print(f"Epoch {epoch:02d}/50 Test Acc: {acc:.2f}%")

    if epoch % 5 == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'test_acc': acc},
                   os.path.join(CKPT_DIR, f'epoch_{epoch:02d}.pt'))
    if acc > best_acc:
        best_acc, best_epoch = acc, epoch
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'test_acc': acc},
                   os.path.join(CKPT_DIR, 'best_stgcn_rjd.pt'))
        print(f" -> New best: {best_acc:.2f}%")

print(f"\nTraining done. Best: {best_acc:.2f}% at epoch {best_epoch}")
print("Proceed to occlusion benchmark on this model.")

Starting ST-GCN-RJD training (50 epochs)...
Epoch 01/50 Test Acc: 13.10%
 -> New best: 13.10%
Epoch 02/50 Test Acc: 16.33%
 -> New best: 16.33%
Epoch 03/50 Test Acc: 21.26%
 -> New best: 21.26%
Epoch 04/50 Test Acc: 22.79%
 -> New best: 22.79%
Epoch 05/50 Test Acc: 25.68%
 -> New best: 25.68%
Epoch 06/50 Test Acc: 28.23%
 -> New best: 28.23%
Epoch 07/50 Test Acc: 28.40%
 -> New best: 28.40%
Epoch 08/50 Test Acc: 32.82%
 -> New best: 32.82%
Epoch 09/50 Test Acc: 36.05%
 -> New best: 36.05%
Epoch 10/50 Test Acc: 31.29%
Epoch 11/50 Test Acc: 31.97%
Epoch 12/50 Test Acc: 36.73%
 -> New best: 36.73%
Epoch 13/50 Test Acc: 37.07%
 -> New best: 37.07%
Epoch 14/50 Test Acc: 38.27%
 -> New best: 38.27%
Epoch 15/50 Test Acc: 39.97%
 -> New best: 39.97%
Epoch 16/50 Test Acc: 40.14%
 -> New best: 40.14%
Epoch 17/50 Test Acc: 41.67%
 -> New best: 41.67%
Epoch 18/50 Test Acc: 41.67%
Epoch 19/50 Test Acc: 44.73%
 -> New best: 44.73%
Epoch 20/50 Test Acc: 42.18%
Epoch 21/50 Test Acc: 43.54%
Epoch 22/50

In [4]:
# Run occlusion benchmark on ST-GCN-RJD (N=30 trials)
import numpy as np
import csv
import os

# Load best RJD checkpoint
rjd_ckpt = torch.load('/content/drive/MyDrive/HRC_Research/checkpoints/stgcn_hri30_rjd_70_10_20/best_stgcn_rjd.pt',
                      map_location=device, weights_only=False)
model.load_state_dict(rjd_ckpt['model_state_dict'])
model.eval()
print(f"RJD model loaded (best acc: {rjd_ckpt['test_acc']:.2f}%)")

def apply_occlusion(data_np, rate, seed=None):
    if rate == 0.0: return data_np.copy()
    if seed is not None: np.random.seed(seed)
    out = data_np.copy()
    V = out.shape[3]
    out[:, :, :, np.random.choice(V, max(1, int(round(V*rate))), replace=False), :] = 0.0
    return out

def run_inference_rjd(model, data_np, labels_np):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for s in range(0, len(labels_np), 64):
            batch = torch.tensor(data_np[s:s+64], dtype=torch.float32).to(device)
            all_preds.append(model(batch).argmax(dim=1).cpu().numpy())
    preds = np.concatenate(all_preds)
    return 100.0 * (preds == labels_np).mean()

OCCLUSION_RATES = [0.0, 0.10, 0.20, 0.30, 0.50]
NUM_TRIALS = 30
rjd_results = []

print("Rate   RJD Mean ± Std")
print("-" * 30)
for rate in OCCLUSION_RATES:
    accs = []
    for trial in range(1 if rate==0.0 else NUM_TRIALS):
        d = apply_occlusion(X_test, rate, seed=42+trial)
        accs.append(run_inference_rjd(model, d, y_test))
    mean_acc, std_acc = float(np.mean(accs)), float(np.std(accs))
    rjd_results.append({'rate': rate, 'mean': mean_acc, 'std': std_acc})
    print(f"{rate*100:.0f}%   {mean_acc:.2f} ± {std_acc:.2f}")

# Save to new CSV
OUT_DIR = '/content/drive/MyDrive/HRC_Research/results/occlusion_benchmark'
os.makedirs(OUT_DIR, exist_ok=True)
csv_path = os.path.join(OUT_DIR, 'rjd_occlusion_results_n30_70_10_20.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['occlusion_rate', 'rjd_mean', 'rjd_std'])
    for r in rjd_results:
        w.writerow([r['rate'], f"{r['mean']:.4f}", f"{r['std']:.4f}"])

print(f"\nSaved -> {csv_path}")
print("Copy these into Table IV as the ST-GCN-RJD column.")

RJD model loaded (best acc: 51.19%)
Rate   RJD Mean ± Std
------------------------------
0%   51.19 ± 0.00
10%   49.04 ± 2.65
20%   46.63 ± 4.20
30%   45.03 ± 4.17
50%   41.29 ± 5.26

Saved -> /content/drive/MyDrive/HRC_Research/results/occlusion_benchmark/rjd_occlusion_results_n30_70_10_20.csv
Copy these into Table IV as the ST-GCN-RJD column.
